# Computer Vision Project (Semester VIII)

## Task: Object Detection (YOLOv8) on Pascal VOC

### Framework
**Ultralytics YOLOv8 (PyTorch)**

### Dataset (online)
**PASCAL VOC 2007 + 2012** via `VOC.yaml` (auto-download + auto-convert to YOLO labels).

### Requirements covered
- Uses **≥ 3 pre-trained models** (YOLOv8n / YOLOv8s / YOLOv8m)
- Uses **Dropout**, **Early Stopping**, and **Weight Decay**, and compares **with vs without** each
- Reports **mAP** and **IoU** (from Ultralytics validation metrics)


In [ ]:
# Install Ultralytics YOLO (includes PyTorch dependency resolution)
# If you're already set up, you can skip this cell.

%pip -q install ultralytics
%pip -q install pandas matplotlib


In [ ]:
import os
from dataclasses import dataclass
from typing import Any, Dict, List

import torch
import pandas as pd
import matplotlib.pyplot as plt

from ultralytics import YOLO

cuda_ok = torch.cuda.is_available()
print("Torch:", torch.__version__)
print("CUDA available:", cuda_ok)
if cuda_ok:
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Note: On Windows, Ultralytics/PyTorch typically uses NVIDIA CUDA GPUs.\n"
          "AMD RX580 usually requires Linux/WSL2 + ROCm; otherwise training will run on CPU.")


In [ ]:
# Dataset note:
# - `VOC.yaml` contains a download+conversion script.
# - When you train with data='VOC.yaml', Ultralytics will download to a datasets directory.
#   Default is usually: ./VOC (relative) or an Ultralytics datasets folder depending on settings.
# Since your YAML uses `path: VOC`, the dataset root will be a folder named `VOC/`.

DATA_YAML = "VOC.yaml"  # you already have this in the project folder

# Quick sanity check: show first lines of YAML path and train/val splits
with open(DATA_YAML, "r", encoding="utf-8") as f:
    for _ in range(25):
        print(f.readline().rstrip())


In [ ]:
DEFAULT_DEVICE = "0" if torch.cuda.is_available() else "cpu"


@dataclass
class ExpConfig:
    imgsz: int = 640
    epochs: int = 50
    batch: int = 16
    device: str = DEFAULT_DEVICE  # "0" for CUDA GPU, otherwise "cpu"
    workers: int = 8

    # Regularization
    dropout: float = 0.2
    weight_decay: float = 5e-4
    patience: int = 20  # early stopping

    project: str = "runs-semester"


cfg = ExpConfig()
print("Using device:", cfg.device)
cfg

In [ ]:
def train_and_collect(model_ckpt: str, run_name: str, overrides: Dict[str, Any]) -> Dict[str, Any]:
    """Train YOLOv8 and return key validation metrics.

    Note: Ultralytics returns a Results object. The exact metric attribute names can vary slightly by version,
    so we extract robustly.
    """
    model = YOLO(model_ckpt)

    args = dict(
        data=DATA_YAML,
        imgsz=cfg.imgsz,
        epochs=cfg.epochs,
        batch=cfg.batch,
        device=cfg.device,
        workers=cfg.workers,
        project=cfg.project,
        name=run_name,
        exist_ok=True,
        plots=True,
        val=True,
        # regularization knobs
        dropout=cfg.dropout,
        weight_decay=cfg.weight_decay,
        patience=cfg.patience,
    )
    args.update(overrides)

    train_res = model.train(**args)

    # Validate using best weights saved in the run folder
    val_res = model.val(data=DATA_YAML, imgsz=cfg.imgsz, batch=cfg.batch, device=cfg.device)

    metrics = getattr(val_res, "metrics", None)

    out = {
        "model": model_ckpt,
        "run": run_name,
        **{f"arg_{k}": v for k, v in args.items() if k in ("dropout", "weight_decay", "patience")},
    }

    # Common metric fields in Ultralytics detection:
    # - metrics.box.map (mAP@0.5:0.95)
    # - metrics.box.map50 (mAP@0.5)
    # - metrics.box.map75
    # - metrics.box.mean_iou (IoU)
    try:
        box = metrics.box
        out["mAP_50_95"] = float(getattr(box, "map"))
        out["mAP_50"] = float(getattr(box, "map50"))
        out["mAP_75"] = float(getattr(box, "map75"))
        if hasattr(box, "mean_iou"):
            out["mean_IoU"] = float(getattr(box, "mean_iou"))
    except Exception:
        # Fallback: store whatever metric dict exists
        if metrics is not None and hasattr(metrics, "results_dict"):
            out.update({f"metric_{k}": float(v) for k, v in metrics.results_dict.items() if isinstance(v, (int, float))})

    return out


In [ ]:
def run_ablation_suite() -> pd.DataFrame:
    # 3 pretrained models (counts as 3 different pre-trained CNN-based detectors)
    models_ = ["yolov8n.pt", "yolov8s.pt", "yolov8m.pt"]

    # Baseline = all regularizers ON
    settings = [
        ("all_on", dict()),
        ("no_dropout", dict(dropout=0.0)),
        # Early stopping OFF: set patience very large so it won't stop early within epochs
        ("no_early_stopping", dict(patience=10_000)),
        ("no_weight_decay", dict(weight_decay=0.0)),
    ]

    rows: List[Dict[str, Any]] = []
    for m in models_:
        for tag, overrides in settings:
            run_name = f"voc_{os.path.splitext(m)[0]}_{tag}"
            print("\n" + "=" * 90)
            print("RUN:", run_name)
            print("OVERRIDES:", overrides)
            print("=" * 90)
            rows.append(train_and_collect(m, run_name, overrides))

    return pd.DataFrame(rows)


# WARNING: This will download VOC (~2.8GB) and train 12 runs.
# For a quick demo on CPU, temporarily set cfg.epochs=5 and use only yolov8n.pt.
results_df = run_ablation_suite()
results_df

In [ ]:
def plot_bars(df: pd.DataFrame, metric: str):
    if metric not in df.columns:
        print(f"Missing metric column: {metric}")
        return

    df = df.copy()
    df["model_short"] = df["model"].str.replace(".pt", "", regex=False)

    pivot = df.pivot(index="model_short", columns="run", values=metric)
    # If you prefer grouping by setting, create a setting column from run names.

    ax = pivot.plot(kind="bar", figsize=(14, 6))
    ax.set_title(metric)
    ax.set_ylabel(metric)
    ax.grid(True, axis="y", alpha=0.3)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


display(results_df)

for metric in ["mAP_50_95", "mAP_50", "mAP_75", "mean_IoU"]:
    plot_bars(results_df, metric)


## Dataset: where it will be downloaded

Your `VOC.yaml` has `path: VOC`, so Ultralytics will create a folder named **`VOC/`** (dataset root) in the working directory (or the configured Ultralytics datasets directory).

After the first training run starts, you should see a structure like:

- `VOC/images/train2007/` + `VOC/labels/train2007/`
- `VOC/images/train2012/` + `VOC/labels/train2012/`
- `VOC/images/test2007/` + `VOC/labels/test2007/`

If you want the dataset to download into a specific place (e.g. `D:/datasets/VOC/`), tell me that path and I’ll update `VOC.yaml` accordingly.

## Discussion / Comparative analysis (what to write)

Compare:
- **Across models**: YOLOv8n vs YOLOv8s vs YOLOv8m (speed/accuracy trade-off)
- **Regularization ablations**:
  - with vs without **Dropout**
  - with vs without **Early Stopping** (patience)
  - with vs without **Weight Decay**

Use the `results_df` table + saved Ultralytics plots to argue why metrics improved or degraded.

## Submission
Submit **only** this notebook: `cv_yolov8_voc.ipynb`
